# Phase 5: Ablation Study & SHAP Explainability



**Best model:** STGTransformer v2 (d_model=128, 100 epochs, full dataset)
**Best checkpoint:** `stgt_v2_continued_best.pt` (epoch 88, val_loss=0.0763)


## Cell 1 — Setup
Mount Drive, load all data, build adjacency matrices.


In [2]:
import os, json, pickle, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import warnings
warnings.filterwarnings('ignore')

LOCAL_FOLDER = './traffic_project'
PROCESSED_DIR = os.path.join(LOCAL_FOLDER, 'processed_data')

# Load config
with open(os.path.join(PROCESSED_DIR, 'config.json')) as f:
    config = json.load(f)

# Load data
print("Loading data...")
data_3d    = np.load(os.path.join(PROCESSED_DIR, 'data_3d.npy'))
adj_tensor = torch.load(os.path.join(PROCESSED_DIR, 'adj_tensor.pt'))

with open(os.path.join(PROCESSED_DIR, 'scaler.pkl'), 'rb') as f:
    scaler = pickle.load(f)

N_SENSORS   = config['n_sensors']
N_TIMES     = config['n_timesteps']
N_FEATURES  = config['n_features']
INPUT_STEPS = config['input_steps']
PRED_STEPS  = config['pred_steps']
BATCH_SIZE  = config['batch_size']
TRAIN_END   = config['train_end_t']
VAL_END     = config['val_end_t']
SPEED_IDX   = 0
FEATURE_COLS = config['feature_cols']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n✅ Device: {device}")

def normalize_adjacency_directed(adj):
    row_sum = adj.sum(dim=1, keepdim=True).clamp(min=1e-8)
    return adj / row_sum

adj_fwd = normalize_adjacency_directed(adj_tensor).to(device)
adj_bwd = normalize_adjacency_directed(adj_tensor.T).to(device)
print(f"✅ All loaded. data_3d: {data_3d.shape}")
print("✅ All imports and data loaded — safe to run any cell")

Loading data...

✅ Device: cuda
✅ All loaded. data_3d: (52116, 325, 31)
✅ All imports and data loaded — safe to run any cell


## Cell 2 — Dataset & DataLoader
Defines `TrafficDataset` — a sliding-window wrapper over the 3D numpy array.

> **Note:** This is a standalone definition identical to Phase 4. Each notebook is self-contained so it can be run independently without loading Phase 4.

Full dataset is used here (no subsampling) since ablation variants train on capped samples internally.

In [3]:
class TrafficDataset(Dataset):
    def __init__(self, data_3d, indices,
                 input_steps=12, pred_steps=12, max_samples=None):
        self.data        = data_3d
        self.input_steps = input_steps
        self.pred_steps  = pred_steps
        if max_samples and len(indices) > max_samples:
            idx = np.random.choice(len(indices), max_samples, replace=False)
            self.indices = [indices[i] for i in idx]
        else:
            self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        t = self.indices[i]
        x = self.data[t - self.input_steps : t]
        y = self.data[t : t + self.pred_steps, :, SPEED_IDX]
        return (torch.tensor(x, dtype=torch.float32),
                torch.tensor(y, dtype=torch.float32))

train_indices = list(range(INPUT_STEPS, TRAIN_END - PRED_STEPS))
val_indices   = list(range(TRAIN_END + INPUT_STEPS, VAL_END - PRED_STEPS))
test_indices  = list(range(VAL_END + INPUT_STEPS, N_TIMES - PRED_STEPS))

BATCH_SIZE = 16

train_dataset = TrafficDataset(data_3d, train_indices)
val_dataset   = TrafficDataset(data_3d, val_indices)
test_dataset  = TrafficDataset(data_3d, test_indices)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

print(f"✅ DataLoaders ready")
print(f"   Train: {len(train_dataset):,} samples ({len(train_loader):,} batches)")
print(f"   Val:   {len(val_dataset):,} samples ({len(val_loader):,} batches)")
print(f"   Test:  {len(test_dataset):,} samples ({len(test_loader):,} batches)")
print(f"   Batch size: {BATCH_SIZE}")


✅ DataLoaders ready
   Train: 36,457 samples (2,279 batches)
   Val:   5,187 samples (325 batches)
   Test:  10,400 samples (650 batches)
   Batch size: 16


## Cell 3 — Model Definition & Load Best Checkpoint
Defines STGTransformer (d_model=128) and loads the best saved weights.

In [4]:
import torch.nn.functional as F

CHECKPOINT_DIR = os.path.join(LOCAL_FOLDER, 'checkpoints')

class DiffusionConvLayer(nn.Module):
    def __init__(self, d_model, n_hops=2):
        super().__init__()
        self.fwd_linears = nn.ModuleList([
            nn.Linear(d_model, d_model, bias=False) for _ in range(n_hops)
        ])
        self.bwd_linears = nn.ModuleList([
            nn.Linear(d_model, d_model, bias=False) for _ in range(n_hops)
        ])
        self.out  = nn.Linear(d_model * 2, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, adj_fwd, adj_bwd):
        h_fwd = x
        for layer in self.fwd_linears:
            h_fwd = F.relu(layer(
                torch.einsum('nm, bmd -> bnd', adj_fwd, h_fwd)
            ))
        h_bwd = x
        for layer in self.bwd_linears:
            h_bwd = F.relu(layer(
                torch.einsum('nm, bmd -> bnd', adj_bwd, h_bwd)
            ))
        combined = torch.cat([h_fwd, h_bwd], dim=-1)
        return self.norm(self.out(combined) + x)


class SpatioTemporalTransformer(nn.Module):
    def __init__(self, n_features, d_model, n_heads, n_gnn_layers,
                 n_tf_layers, n_sensors, pred_steps, dropout=0.1):
        super().__init__()
        self.n_sensors  = n_sensors
        self.pred_steps = pred_steps
        self.d_model    = d_model

        self.input_proj = nn.Linear(n_features, d_model)
        self.gnn_layers = nn.ModuleList([
            DiffusionConvLayer(d_model) for _ in range(n_gnn_layers)
        ])
        self.spatial_gate = nn.Sequential(
            nn.Linear(d_model * 2, d_model), nn.Sigmoid()
        )
        self.pos_embedding = nn.Parameter(
            torch.randn(1, INPUT_STEPS, d_model) * 0.02
        )
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=n_tf_layers
        )
        self.tf_norm   = nn.LayerNorm(d_model)
        self.pred_head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, pred_steps)
        )

    def forward(self, x, adj_fwd, adj_bwd):
        B, T, N, n_feat = x.shape
        x      = self.input_proj(x)
        x_orig = x.clone()
        x_flat = x.reshape(B * T, N, self.d_model)
        for gnn in self.gnn_layers:
            x_flat = gnn(x_flat, adj_fwd, adj_bwd)
        x_spatial = x_flat.reshape(B, T, N, self.d_model)
        gate = self.spatial_gate(torch.cat([x_spatial, x_orig], dim=-1))
        x    = gate * x_spatial + (1 - gate) * x_orig
        x    = x.permute(0, 2, 1, 3).reshape(B * N, T, self.d_model)
        x    = x + self.pos_embedding[:, :T, :]
        x    = self.transformer(x)
        x    = self.tf_norm(x[:, -1, :])
        pred = self.pred_head(x)
        pred = pred.reshape(B, N, self.pred_steps)
        pred = pred.permute(0, 2, 1)
        return pred


# Instantiate model matching checkpoint architecture (STGTransformer v2: d_model=128, n_heads=4)
model = SpatioTemporalTransformer(
    n_features   = N_FEATURES,
    d_model      = 128,
    n_heads      = 4,
    n_gnn_layers = 2,
    n_tf_layers  = 2,
    n_sensors    = N_SENSORS,
    pred_steps   = PRED_STEPS,
    dropout      = 0.1
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ SpatioTemporalTransformer ready (d_model=128)")
print(f"   Parameters: {total_params:,}")

# Load best checkpoint (STGTransformer v2, continued 100-epoch run)
ckpt = torch.load(os.path.join(CHECKPOINT_DIR, 'stgt_v2_continued_best.pt'), map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"✅ Best checkpoint loaded (epoch={ckpt['epoch']}, val_loss={ckpt['val_loss']:.4f})")


✅ SpatioTemporalTransformer ready (d_model=128)
   Parameters: 641,740
✅ Best checkpoint loaded (epoch=88, val_loss=0.0763)


## Cell 4 — Ablation Helper Functions
Defines `TrafficDatasetMasked` and `train_ablation()` for ablation experiments.

In [9]:
import os
import json
import time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

LOCAL_FOLDER = './traffic_project'
RESULTS_DIR  = os.path.join(LOCAL_FOLDER, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Section 1: Masked Dataset ──────────────────────────────────────────────
class TrafficDatasetMasked(Dataset):
    """Dataset with feature masking for ablation study."""
    def __init__(self, data_3d, indices, feature_indices,
                 input_steps=12, pred_steps=12, max_samples=None):
        self.data            = data_3d
        self.feature_indices = feature_indices
        self.input_steps     = input_steps
        self.pred_steps      = pred_steps
        self.n_features      = data_3d.shape[2]
        if max_samples and len(indices) > max_samples:
            idx = np.random.choice(len(indices), max_samples, replace=False)
            self.indices = [indices[i] for i in idx]
        else:
            self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        t = self.indices[i]
        x = self.data[t - self.input_steps : t].copy()
        mask = np.zeros(self.n_features, dtype=np.float32)
        mask[self.feature_indices] = 1.0
        x = x * mask[np.newaxis, np.newaxis, :]
        y = self.data[t : t + self.pred_steps, :, SPEED_IDX]
        return (torch.tensor(x, dtype=torch.float32),
                torch.tensor(y, dtype=torch.float32))


# ── Section 2: Model Factory ────────────────────────────────────────────────
def build_model():
    """Build a fresh model instance for ablation training."""
    return SpatioTemporalTransformer(
        n_features=N_FEATURES, d_model=128, n_heads=8,
        n_gnn_layers=2, n_tf_layers=2, n_sensors=N_SENSORS,
        pred_steps=PRED_STEPS, dropout=0.1
    ).to(device)


# ── Section 3: Ablation Training Loop ───────────────────────────────────────
def train_ablation(feature_indices, label, n_epochs=25,
                   train_samples=8000, val_samples=2000):
    """Train one ablation variant from scratch."""
    print(f"\n{'='*55}")
    print(f"Training: {label}")
    print(f"Features ({len(feature_indices)}): {[FEATURE_COLS[i] for i in feature_indices]}")
    print(f"{'='*55}")

    abl_model = build_model()
    optimizer = torch.optim.Adam(abl_model.parameters(), lr=5e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3
    )
    loss_fn       = nn.HuberLoss(delta=1.0)
    best_val_loss = float('inf')
    best_state    = None
    patience      = 0
    EARLY_STOP    = 6

    for epoch in range(1, n_epochs + 1):
        t0 = time.time()
        train_ds = TrafficDatasetMasked(data_3d, train_indices,
                                        feature_indices, max_samples=train_samples)
        # Fixed: num_workers=0 prevents Windows IPC serialization errors
        train_ld = DataLoader(train_ds, batch_size=BATCH_SIZE,
                              shuffle=True, num_workers=0, pin_memory=True)
        val_ds   = TrafficDatasetMasked(data_3d, val_indices,
                                        feature_indices, max_samples=val_samples)
        val_ld   = DataLoader(val_ds, batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=0, pin_memory=True)

        abl_model.train()
        train_loss = 0
        for xb, yb in train_ld:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            pred = abl_model(xb, adj_fwd, adj_bwd)
            loss = loss_fn(pred, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(abl_model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()
        avg_train = train_loss / len(train_ld)

        abl_model.eval()
        val_loss = 0
        with torch.no_grad():
            for xb, yb in val_ld:
                xb, yb = xb.to(device), yb.to(device)
                val_loss += loss_fn(abl_model(xb, adj_fwd, adj_bwd), yb).item()
        avg_val = val_loss / len(val_ld)
        print(f"  Epoch {epoch:2d}/{n_epochs}  train={avg_train:.4f}  val={avg_val:.4f}  ({time.time()-t0:.0f}s)")

        scheduler.step(avg_val)
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            best_state    = {k: v.clone() for k, v in abl_model.state_dict().items()}
            patience      = 0
        else:
            patience += 1
            if patience >= EARLY_STOP:
                print(f"  Early stopping at epoch {epoch}")
                break

    abl_model.load_state_dict(best_state)
    abl_model.eval()
    test_ds = TrafficDatasetMasked(data_3d, test_indices, feature_indices)
    test_ld = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=0, pin_memory=True)
    all_preds, all_trues = [], []
    with torch.no_grad():
        for xb, yb in test_ld:
            pred = abl_model(xb.to(device), adj_fwd, adj_bwd)
            all_preds.append(pred.cpu().numpy())
            all_trues.append(yb.cpu().numpy())
    all_preds = np.concatenate(all_preds, axis=0)
    all_trues = np.concatenate(all_trues, axis=0)

    results = {}
    for h, horizon in [(2,'15 min'),(5,'30 min'),(11,'60 min')]:
        mae = float(np.mean(np.abs(
            all_trues[:,h,:].flatten() - all_preds[:,h,:].flatten()
        )))
        results[horizon] = mae
        print(f"  Test MAE @ {horizon}: {mae:.4f}")
    return results

print("✅ train_ablation() function ready")


# ── Section 4: Run Ablation Experiments & Save File ───────────────────────
variants = {
    'Speed + Time only': [0, 1, 2, 3],
    'Speed + Time + Weather': [0, 1, 2, 3, 4, 5, 6, 7, 8],
    'Full (Speed + Time + Weather + Accidents)': list(range(N_FEATURES))
}

proper_ablation_results = {}
for label, feat_idxs in variants.items():
    res = train_ablation(feat_idxs, label, n_epochs=20)
    proper_ablation_results[label] = res

# Export results to results directory
output_file = os.path.join(RESULTS_DIR, 'proper_ablation_results.json')
with open(output_file, 'w') as f:
    json.dump(proper_ablation_results, f, indent=2)

print(f"\n✅ Saved ablation results to '{output_file}'")

✅ train_ablation() function ready

Training: Speed + Time only
Features (4): ['speed', 'hour_sin', 'hour_cos', 'dow_sin']
  Epoch  1/20  train=0.0948  val=0.0876  (92s)
  Epoch  2/20  train=0.0771  val=0.0869  (91s)
  Epoch  3/20  train=0.0726  val=0.0847  (91s)
  Epoch  4/20  train=0.0722  val=0.0844  (91s)
  Epoch  5/20  train=0.0713  val=0.0839  (91s)
  Epoch  6/20  train=0.0703  val=0.0861  (91s)
  Epoch  7/20  train=0.0708  val=0.0839  (91s)
  Epoch  8/20  train=0.0711  val=0.0804  (91s)
  Epoch  9/20  train=0.0716  val=0.0843  (91s)
  Epoch 10/20  train=0.0717  val=0.0781  (91s)
  Epoch 11/20  train=0.0711  val=0.0833  (91s)
  Epoch 12/20  train=0.0705  val=0.0846  (91s)
  Epoch 13/20  train=0.0710  val=0.0809  (91s)
  Epoch 14/20  train=0.0707  val=0.0838  (90s)
  Epoch 15/20  train=0.0689  val=0.0783  (91s)
  Epoch 16/20  train=0.0687  val=0.0811  (91s)
  Early stopping at epoch 16
  Test MAE @ 15 min: 0.1579
  Test MAE @ 30 min: 0.2046
  Test MAE @ 60 min: 0.2622

Training: Sp

## Cell 5 — Ablation Results
Loads pre-computed ablation results from Drive and prints comparison table.

In [5]:
# Define path location based on organized directory structure
LOCAL_FOLDER = './traffic_project'
RESULTS_DIR  = os.path.join(LOCAL_FOLDER, 'results')

# Ablation results already computed and saved from previous run
with open(os.path.join(RESULTS_DIR, 'proper_ablation_results.json')) as f:
    proper_ablation_results = json.load(f)

# Show actual keys so we know what's in the file
print("Keys in ablation results:")
for key in proper_ablation_results:
    print(f"  '{key}'")

speed_std = 9.44

print("\n" + "="*75)
print("ABLATION STUDY: Impact of Multi-Modal Features")
print("="*75)
print(f"{'Variant':<45} {'15 min':>8} {'30 min':>8} {'60 min':>8}")
print("-"*75)

# Print all variants using actual keys
for label, res in proper_ablation_results.items():
    m15 = res['15 min']
    m30 = res['30 min']
    m60 = res['60 min']
    marker = ' ← full' if 'Accident' in label or 'full' in label.lower() else ''
    print(f"  {label:<43} {m15:>8.4f} {m30:>8.4f} {m60:>8.4f}{marker}")

print("="*75)

# Compute gain — first variant is baseline, last is full
keys   = list(proper_ablation_results.keys())
base   = proper_ablation_results[keys[0]]
full   = proper_ablation_results[keys[-1]]

for horizon in ['15 min', '30 min', '60 min']:
    gain = (base[horizon] - full[horizon]) / base[horizon] * 100
    print(f"  Multi-modal gain @ {horizon}: {gain:.1f}%")

Keys in ablation results:
  'Speed + Time only'
  'Speed + Time + Weather'
  'Full (Speed + Time + Weather + Accidents)'

ABLATION STUDY: Impact of Multi-Modal Features
Variant                                         15 min   30 min   60 min
---------------------------------------------------------------------------
  Speed + Time only                             0.1579   0.2046   0.2622
  Speed + Time + Weather                        0.1571   0.2052   0.2628
  Full (Speed + Time + Weather + Accidents)     0.1659   0.2133   0.2682 ← full
  Multi-modal gain @ 15 min: -5.1%
  Multi-modal gain @ 30 min: -4.2%
  Multi-modal gain @ 60 min: -2.3%


In [6]:
import os
import numpy as np
import torch

LOCAL_FOLDER = './traffic_project'
RESULTS_DIR  = os.path.join(LOCAL_FOLDER, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── 1. SELECT REPRESENTATIVE TEST SAMPLES ─────────────────────────────────────
np.random.seed(42)
N_SHAP_SAMPLES = 200
sample_test_indices = np.random.choice(test_indices, size=N_SHAP_SAMPLES, replace=False)

print(f"Computing feature attributions for {N_SHAP_SAMPLES} samples at 15-min horizon...")

model.eval()
shap_values_list = []
HORIZON_IDX = 2  # Index 2 corresponds to 15-min forecast (t + 3 steps @ 5-min intervals)

# ── 2. COMPUTE GRADIENT-BASED INPUT ATTRIBUTIONS ─────────────────────────────
for i, idx in enumerate(sample_test_indices, 1):
    # Slice input window: Shape (1, T=12, N=325, F=31)
    x_sample = data_3d[idx - INPUT_STEPS : idx]
    x_tensor = torch.tensor(x_sample, dtype=torch.float32, device=device).unsqueeze(0)
    x_tensor.requires_grad = True

    # Forward pass
    pred = model(x_tensor, adj_fwd, adj_bwd)  # Shape: (1, pred_steps=12, N_sensors=325)
    
    # Target: Predicted speed across all network sensors at 15-min horizon
    target = pred[:, HORIZON_IDX, :].sum()
    
    # Backward pass to compute input feature gradients
    model.zero_grad()
    target.backward()

    # Input * Gradient attribution: Shape (1, T, N, F)
    attr = (x_tensor.grad * x_tensor).detach()
    
    # Average attribution magnitude across all time steps and road sensors -> Shape: (F,)
    sample_shap = attr.abs().mean(dim=(0, 1, 2)).cpu().numpy()
    shap_values_list.append(sample_shap)
    
    if i % 50 == 0 or i == N_SHAP_SAMPLES:
        print(f"  Processed [{i:>3}/{N_SHAP_SAMPLES}] samples")

# Aggregate into matrix: Shape (200, N_FEATURES)
shap_values = np.array(shap_values_list)

# ── 3. STORE OUTPUT ARTIFACT ──────────────────────────────────────────────────
output_shap_path = os.path.join(RESULTS_DIR, 'shap_values.npy')
np.save(output_shap_path, shap_values)
print(f"\n✅ Successfully generated and saved SHAP array {shap_values.shape} to:\n   '{output_shap_path}'")

Computing feature attributions for 200 samples at 15-min horizon...
  Processed [ 50/200] samples
  Processed [100/200] samples
  Processed [150/200] samples
  Processed [200/200] samples

✅ Successfully generated and saved SHAP array (200, 31) to:
   './traffic_project\results\shap_values.npy'


## Cell 6 — SHAP Feature Importance Charts
Loads saved SHAP values and generates feature importance visualizations.

In [7]:
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# Path configuration based on project structure
LOCAL_FOLDER = './traffic_project'
RESULTS_DIR  = os.path.join(LOCAL_FOLDER, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

# Load saved SHAP values
shap_path = os.path.join(RESULTS_DIR, 'shap_values.npy')
if not os.path.exists(shap_path):
    fallback_path = os.path.join(LOCAL_FOLDER, 'shap_values.npy')
    if os.path.exists(fallback_path):
        shap_path = fallback_path
    else:
        raise FileNotFoundError(
            f"❌ File not found at '{shap_path}' or '{fallback_path}'. "
            f"Ensure 'shap_values.npy' is placed in your results folder."
        )

shap_values = np.load(shap_path)

name_map = {
    'speed':             'Traffic Speed',
    'hour':              'Hour of Day',
    'dayofweek':         'Day of Week',
    'is_weekend':        'Is Weekend',
    'Temperature(F)':    'Temperature',
    'Humidity(%)':       'Humidity',
    'Visibility(mi)':    'Visibility',
    'Wind_Speed(mph)':   'Wind Speed',
    'weather_code':      'Weather Condition',
    'acc_count_60min':   'Accident Count (60 min)',
    'acc_max_severity':  'Accident Severity',
    'acc_mins_since':    'Mins Since Accident',
}
display_names = [name_map.get(c, c) for c in FEATURE_COLS]

mean_shap   = np.mean(np.abs(shap_values), axis=0)
sorted_idx  = np.argsort(mean_shap)[::-1]
sorted_vals = mean_shap[sorted_idx]
sorted_disp = [display_names[i] for i in sorted_idx]

colors = ['#D85A30' if 'Accident' in n or 'Mins Since' in n
          else '#27AE60' if n in
              ['Temperature', 'Humidity', 'Visibility',
               'Wind Speed', 'Weather Condition']
          else '#378ADD'
          for n in sorted_disp]

# Chart 1: Feature importance bar chart
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(len(sorted_disp)), sorted_vals, color=colors)
ax.set_yticks(range(len(sorted_disp)))
ax.set_yticklabels(sorted_disp, fontsize=11)
ax.set_xlabel('Mean |SHAP Value|', fontsize=12)
ax.set_title('Feature Importance — STGTransformer\n'
             '(15-min prediction, 200 test samples)',
             fontsize=13, fontweight='bold')

ax.legend(handles=[
    Patch(color='#378ADD', label='Traffic & time'),
    Patch(color='#27AE60', label='Weather'),
    Patch(color='#D85A30', label='Accident'),
], loc='lower right', fontsize=10)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'shap_importance.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ SHAP feature importance chart saved")

# Feature importance ranking
print(f"\nFeature importance ranking:")
for rank, (name, val) in enumerate(zip(sorted_disp, sorted_vals), 1):
    bar = '█' * int(val / sorted_vals[0] * 30)
    print(f"  {rank:2d}. {name:<25} {val:.5f}  {bar}")

# Feature group breakdown
traffic_cols  = [i for i, n in enumerate(display_names)
                 if n in ['Traffic Speed', 'Hour of Day', 'Day of Week', 'Is Weekend']]
weather_cols  = [i for i, n in enumerate(display_names)
                 if n in ['Temperature', 'Humidity', 'Visibility', 'Wind Speed', 'Weather Condition']]
accident_cols = [i for i, n in enumerate(display_names)
                 if 'Accident' in n or 'Mins Since' in n]

group_importance = {
    'Traffic & Time': float(np.sum(mean_shap[traffic_cols])),
    'Weather':        float(np.sum(mean_shap[weather_cols])),
    'Accident':       float(np.sum(mean_shap[accident_cols])),
}
total = sum(group_importance.values())
print(f"\nFeature group breakdown:")
for grp, imp in group_importance.items():
    print(f"  {grp:<20} {imp:.5f}  ({imp/total*100:.1f}%)")

✅ SHAP feature importance chart saved

Feature importance ranking:
   1. Traffic Speed             0.05621  ██████████████████████████████
   2. weather_Clear             0.01672  ████████
   3. hour_cos                  0.01587  ████████
   4. hour_sin                  0.01256  ██████
   5. weather_Overcast          0.00940  █████
   6. weather_Mostly Cloudy     0.00777  ████
   7. weather_Partly Cloudy     0.00767  ████
   8. Is Weekend                0.00657  ███
   9. weather_Scattered Clouds  0.00430  ██
  10. weather_Light Rain        0.00337  █
  11. Temperature               0.00238  █
  12. dow_sin                   0.00181  
  13. weather_Unknown           0.00151  
  14. dow_cos                   0.00113  
  15. Humidity                  0.00108  
  16. Wind Speed                0.00106  
  17. Accident Severity         0.00073  
  18. weather_Rain              0.00065  
  19. Accident Count (60 min)   0.00060  
  20. weather_Haze              0.00051  
  21. Mins Since Acci

## Cell 7 — Final Summary
Prints complete results table (our model vs published benchmarks) and Drive file inventory.

In [10]:
# Final results summary

speed_std = 9.44

print("=" * 70)
print("PHASE 5 — FINAL RESULTS SUMMARY")
print("=" * 70)

print("\n── Model Comparison (MAE in mph) ──")
print(f"{'Model':<30} {'15 min':>8} {'30 min':>8} {'60 min':>8}")
print("-" * 58)

results_mph = {
    'Historical Average':      [2.93, 3.11, 3.54],
    'ARIMA':                   [2.98, 2.99, 3.03],
    'LSTM (no spatial)':       [1.47, 1.97, 2.61],
    'STGTransformer (ours)':   [1.46, 1.91, 2.42],
    '--- Published models ---': [None, None, None],
    'DCRNN (2018)':            [1.38, 1.74, 2.07],
    'STGCN (2018)':            [1.36, 1.81, 2.49],
    'Graph WaveNet (2019)':    [1.30, 1.63, 1.95],
}

for name, vals in results_mph.items():
    if vals[0] is None:
        print(f"  {name}")
        continue
    marker = ' ← OURS' if 'ours' in name.lower() else ''
    print(f"  {name:<28} {vals[0]:>7.2f}  {vals[1]:>7.2f}  {vals[2]:>7.2f}{marker}")

print("=" * 70)
print("\n── Key Findings ──")
print("  ✅ STGTransformer beats STGCN at 60 min (2.42 vs 2.49 mph)")
print("  ✅ Within 5.8% of DCRNN at 15 min (1.46 vs 1.38 mph)")
print("  ✅ Beats LSTM at all horizons (spatial modeling helps)")
print("  ⚠️  Ablation study: adding weather + accident features did NOT improve")
print("     accuracy — Speed+Time-only scored best (MAE 0.158/0.205/0.262 @")
print("     15/30/60 min) vs the Full multi-modal variant (0.166/0.213/0.268),")
print("     i.e. 2-5% worse across horizons. Multi-modal fusion is a modeling")
print("     capability, not a validated accuracy gain, on this dataset/setup.")
print("  ✅ SHAP confirms this: Traffic & Time features drive 90.6% of prediction")
print("     attribution, Weather only 7.0%, Accidents just 2.4% — traffic speed")
print("     and time-of-day dominate, weather/accident signal is weak/noisy")
print("  ✅ Novel: SHAP explainability integrated (gradient-based attribution)")
print("  ✅ Best model: stgt_v2_continued_best.pt (epoch 88, val_loss=0.0763)")

print("\n✅ Phase 5 complete — Ready for Phase 6: FastAPI + Docker + Streamlit")


PHASE 5 — FINAL RESULTS SUMMARY

── Model Comparison (MAE in mph) ──
Model                            15 min   30 min   60 min
----------------------------------------------------------
  Historical Average              2.93     3.11     3.54
  ARIMA                           2.98     2.99     3.03
  LSTM (no spatial)               1.47     1.97     2.61
  STGTransformer (ours)           1.46     1.91     2.42 ← OURS
  --- Published models ---
  DCRNN (2018)                    1.38     1.74     2.07
  STGCN (2018)                    1.36     1.81     2.49
  Graph WaveNet (2019)            1.30     1.63     1.95

── Key Findings ──
  ✅ STGTransformer beats STGCN at 60 min (2.42 vs 2.49 mph)
  ✅ Within 5.8% of DCRNN at 15 min (1.46 vs 1.38 mph)
  ✅ Beats LSTM at all horizons (spatial modeling helps)
  ⚠️  Ablation study: adding weather + accident features did NOT improve
     accuracy — Speed+Time-only scored best (MAE 0.158/0.205/0.262 @
     15/30/60 min) vs the Full multi-modal varian